# Modelado y Join -> Capa Gold
Creación de tablón analítico y particionamiento.

In [ ]:
import pyspark.sql.functions as F

base_path = 'file:/Workspace/Repos/your_email/castor-data-engineer/data'

try:
    df_user = spark.read.format('delta').load(f'{base_path}/silver/usuarios')
    df_trans = spark.read.format('delta').load(f'{base_path}/silver/transacciones')
except:
    df_user = spark.read.parquet(f'{base_path}/silver/usuarios')
    df_trans = spark.read.parquet(f'{base_path}/silver/transacciones')

In [ ]:
# Join con Broadcast en la tabla pequeña
df_join = df_trans.join(F.broadcast(df_user), on='id_usuario', how='left')

In [ ]:
# Agregaciones
df_gold = df_join.groupBy('id_usuario', F.year('fecha_transaccion').alias('year'), F.month('fecha_transaccion').alias('month')) \
                 .agg(
                     F.sum('monto_clean').alias('total_gastado'),
                     F.count('id_transaccion').alias('num_transacciones'),
                     F.avg('monto_clean').alias('ticket_promedio')
                 )

In [ ]:
# Guardar en Gold particionado
try:
    df_gold.write.format('delta').mode('overwrite').partitionBy('year', 'month').save(f'{base_path}/gold/customer_summary')
except:
    df_gold.write.mode('overwrite').partitionBy('year', 'month').parquet(f'{base_path}/gold/customer_summary')